# Churn model, customer value (CLV), and acquisition targeting

This notebook is the **main** workflow:

1. Train a churn model (probability a customer is “at risk”).
2. Turn that risk into a simple **lifetime value** score (ranking customers).
3. Simulate acquisition targeting and compare **top-value targeting** vs **random** selection.

**Data:** path from `config.yaml` (`data.customer_training_csv`; default includes a public-health proxy from scraped Wikipedia state tables) (same file as `01_preprocessing.ipynb`). See that notebook for a plain-English explanation of where it comes from.

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

os.environ.setdefault("MPLBACKEND", "Agg")
%matplotlib inline

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

CFG = yaml.safe_load((ROOT / "config.yaml").read_text(encoding="utf-8"))
SEED = int(CFG["project"]["random_seed"])
OUT_DIR = ROOT / str(CFG["project"]["output_dir"])
OUT_DIR.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(SEED)

DATA_REL = str(CFG.get("data", {}).get("customer_training_csv", "data/customers.csv"))
DATA_PATH = Path(DATA_REL) if Path(DATA_REL).is_absolute() else (ROOT / DATA_REL)
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple

import numpy as np
import pandas as pd
import yaml
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC_FEATURES: List[str] = [
    "age",
    "tenure_months",
    "num_orders",
    "total_spend",
    "avg_order_value",
    "days_since_last_order",
    "email_opens_30d",
    "app_sessions_30d",
    "public_uninsured_rate_proxy",
]
CATEGORICAL_FEATURES: List[str] = ["region"]
TARGET_COLUMN = "churned"
INTERNAL_COLUMNS: Tuple[str, ...] = ("_acquisition_prob", "_monthly_spend_intensity")


@dataclass
class TrainValTestSplit:
    X_train: pd.DataFrame
    X_val: pd.DataFrame
    X_test: pd.DataFrame
    y_train: np.ndarray
    y_val: np.ndarray
    y_test: np.ndarray
    acquisition_prob_train: np.ndarray
    acquisition_prob_val: np.ndarray
    acquisition_prob_test: np.ndarray


def make_preprocessor() -> ColumnTransformer:
    numeric_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, NUMERIC_FEATURES),
            ("cat", categorical_pipe, CATEGORICAL_FEATURES),
        ]
    )


def temporal_or_random_split(
    df: pd.DataFrame,
    train_fraction: float,
    val_fraction: float,
    random_seed: int,
) -> TrainValTestSplit:
    if not 0 < train_fraction < 1:
        raise ValueError("train_fraction must be in (0,1)")
    if not 0 < val_fraction < 1:
        raise ValueError("val_fraction must be in (0,1)")
    if train_fraction + val_fraction >= 1:
        raise ValueError("train_fraction + val_fraction must be < 1")

    work = df.reset_index(drop=True)
    idx = np.arange(len(work))
    rng = np.random.default_rng(random_seed)
    rng.shuffle(idx)

    n = len(idx)
    n_train = int(n * train_fraction)
    n_val = int(n * val_fraction)

    train_idx = idx[:n_train]
    val_idx = idx[n_train : n_train + n_val]
    test_idx = idx[n_train + n_val :]

    def _pack(indices: np.ndarray) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray]:
        part = work.iloc[indices]
        y = part[TARGET_COLUMN].to_numpy()
        latent = part["_acquisition_prob"].to_numpy()
        drop_cols = list(INTERNAL_COLUMNS) + [TARGET_COLUMN]
        if "customer_id" in part.columns:
            drop_cols.append("customer_id")
        X = part.drop(columns=drop_cols)
        return X, y, latent

    X_train, y_train, a_train = _pack(train_idx)
    X_val, y_val, a_val = _pack(val_idx)
    X_test, y_test, a_test = _pack(test_idx)

    return TrainValTestSplit(
        X_train=X_train,
        X_val=X_val,
        X_test=X_test,
        y_train=y_train,
        y_val=y_val,
        y_test=y_test,
        acquisition_prob_train=a_train,
        acquisition_prob_val=a_val,
        acquisition_prob_test=a_test,
    )

split = temporal_or_random_split(
    df,
    train_fraction=float(CFG['data']['train_fraction']),
    val_fraction=float(CFG['data']['val_fraction']),
    random_seed=SEED,
)


## Train / validation / test sizes and class balance

Before fitting the churn model, confirm how many rows each split has and how imbalanced the churn label is.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sizes = {"train": len(split.y_train), "val": len(split.y_val), "test": len(split.y_test)}
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.barplot(x=list(sizes.keys()), y=list(sizes.values()), ax=axes[0])
axes[0].set_title("Rows per split")

bal = pd.Series({"not churned": (split.y_train == 0).sum(), "churned": (split.y_train == 1).sum()})
axes[1].pie(bal.values, labels=bal.index, autopct="%1.1f%%", startangle=90)
axes[1].set_title("Train-set churn mix")

plt.tight_layout()
plt.show()

print("Training rows:", sizes["train"], "| val:", sizes["val"], "| test:", sizes["test"])
print("Train churn rate:", float(split.y_train.mean()))


In [ ]:

from dataclasses import dataclass
from typing import Any, Dict, Literal, Optional

from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

try:
    from xgboost import XGBClassifier  # type: ignore
except Exception:
    XGBClassifier = None  # type: ignore

ModelType = Literal["gradient_boosting", "logistic", "random_forest", "xgboost"]


@dataclass
class ChurnModelConfig:
    model_type: ModelType = "gradient_boosting"
    n_estimators: int = 120
    max_depth: int = 4
    learning_rate: float = 0.08
    min_samples_leaf: int = 20
    class_weight: Optional[str] = "balanced"
    calibration_cv: int = 3


def _build_classifier(cfg: ChurnModelConfig, random_seed: int):
    if cfg.model_type == "logistic":
        return LogisticRegression(
            max_iter=2000,
            class_weight=cfg.class_weight,
            random_state=random_seed,
            solver="lbfgs",
        )
    if cfg.model_type == "random_forest":
        return RandomForestClassifier(
            n_estimators=max(cfg.n_estimators, 50),
            max_depth=cfg.max_depth,
            min_samples_leaf=cfg.min_samples_leaf,
            class_weight=cfg.class_weight,
            random_state=random_seed,
            n_jobs=-1,
        )
    if cfg.model_type == "gradient_boosting":
        return GradientBoostingClassifier(
            random_state=random_seed,
            n_estimators=cfg.n_estimators,
            max_depth=cfg.max_depth,
            learning_rate=cfg.learning_rate,
            min_samples_leaf=cfg.min_samples_leaf,
        )
    if cfg.model_type == "xgboost":
        if XGBClassifier is None:
            raise ImportError("Install xgboost to use model_type='xgboost'.")
        return XGBClassifier(
            n_estimators=cfg.n_estimators,
            max_depth=cfg.max_depth,
            learning_rate=cfg.learning_rate,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            random_state=random_seed,
            n_jobs=-1,
            eval_metric="logloss",
        )
    raise ValueError(f"Unknown model_type: {cfg.model_type}")


def build_churn_pipeline(cfg: ChurnModelConfig, random_seed: int) -> Pipeline:
    clf = _build_classifier(cfg, random_seed)
    return Pipeline(steps=[("prep", make_preprocessor()), ("clf", clf)])


def fit_churn_model(
    cfg: ChurnModelConfig,
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    random_seed: int,
    calibrate: bool = True,
) -> Any:
    base = build_churn_pipeline(cfg, random_seed)
    if not calibrate:
        base.fit(X_train, y_train)
        return base
    calibrated = CalibratedClassifierCV(estimator=base, method="isotonic", cv=cfg.calibration_cv)
    calibrated.fit(X_train, y_train)
    return calibrated


def predict_churn_proba(model: Any, X: pd.DataFrame) -> np.ndarray:
    proba = model.predict_proba(X)
    return proba[:, 1]


In [ ]:
churn_cfg = ChurnModelConfig(
    model_type=str(CFG['churn_model']['model_type']),
    n_estimators=int(CFG['churn_model']['n_estimators']),
    max_depth=int(CFG['churn_model']['max_depth']),
    learning_rate=float(CFG['churn_model']['learning_rate']),
    min_samples_leaf=int(CFG['churn_model']['min_samples_leaf']),
    class_weight=CFG['churn_model'].get('class_weight'),
)

churn_model = fit_churn_model(
    churn_cfg,
    split.X_train,
    split.y_train,
    random_seed=SEED,
    calibrate=True,
)

y_score_full_test = predict_churn_proba(churn_model, split.X_test)
y_score_full_test[:5]

In [ ]:

from dataclasses import dataclass


@dataclass
class CLVModelConfig:
    margin_rate: float = 0.25
    discount_monthly: float = 0.002
    max_horizon_months: int = 60
    min_churn_prob: float = 0.001
    max_churn_prob: float = 0.999


def _historical_monthly_margin(df: pd.DataFrame, margin_rate: float) -> np.ndarray:
    tenure = np.maximum(df["tenure_months"].to_numpy(dtype=float), 1.0)
    spend = df["total_spend"].to_numpy(dtype=float)
    return (spend * margin_rate) / tenure


def expected_discounted_future_margin(
    monthly_margin: np.ndarray,
    churn_prob: np.ndarray,
    cfg: CLVModelConfig,
) -> np.ndarray:
    p = np.clip(churn_prob, cfg.min_churn_prob, cfg.max_churn_prob)
    d = cfg.discount_monthly
    m = monthly_margin

    t = np.arange(1, cfg.max_horizon_months + 1, dtype=float)
    survival = np.power(1.0 - p[:, None], t - 1.0)
    discount = np.power(1.0 + d, t)
    weights = survival / discount[None, :]
    return (m[:, None] * weights).sum(axis=1)


def estimate_clv(
    X_row_aligned: pd.DataFrame,
    churn_prob: np.ndarray,
    cfg: CLVModelConfig,
) -> pd.Series:
    hist = _historical_monthly_margin(X_row_aligned, cfg.margin_rate) * X_row_aligned[
        "tenure_months"
    ].to_numpy(dtype=float)
    future = expected_discounted_future_margin(
        monthly_margin=_historical_monthly_margin(X_row_aligned, cfg.margin_rate),
        churn_prob=churn_prob,
        cfg=cfg,
    )
    clv = hist + future
    return pd.Series(clv, index=X_row_aligned.index, name="pred_clv")


In [ ]:

from dataclasses import dataclass
from typing import Dict

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, roc_curve


@dataclass
class ChurnMetrics:
    roc_auc: float
    average_precision: float


def churn_discrimination_metrics(y_true: np.ndarray, y_score: np.ndarray) -> ChurnMetrics:
    return ChurnMetrics(
        roc_auc=float(roc_auc_score(y_true, y_score)),
        average_precision=float(average_precision_score(y_true, y_score)),
    )


def plot_churn_model_performance(y_true: np.ndarray, y_score: np.ndarray, out_path: Path, title: str) -> None:
    fpr, tpr, _ = roc_curve(y_true, y_score)
    prec, rec, _ = precision_recall_curve(y_true, y_score)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    axes[0].plot(fpr, tpr, label="ROC")
    axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
    axes[0].set_title("ROC curve")
    axes[0].set_xlabel("False positive rate")
    axes[0].set_ylabel("True positive rate")

    axes[1].plot(rec, prec, label="PR")
    axes[1].set_title("Precision-recall curve")
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")

    for ax in axes:
        ax.grid(True, alpha=0.3)

    fig.suptitle(title)
    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def plot_clv_distribution(clv: np.ndarray, out_path: Path, title: str) -> None:
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.histplot(clv, kde=True, ax=ax, bins=40, color="#2c7fb8")
    ax.set_title(title)
    ax.set_xlabel("Predicted CLV")
    ax.set_ylabel("Customers")
    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def plot_lift_chart(y_true: np.ndarray, scores: np.ndarray, out_path: Path, n_bins: int = 10, title: str = "") -> None:
    order = np.argsort(-scores)
    y_sorted = y_true[order]
    n = len(y_sorted)
    baseline = y_sorted.mean()

    lifts = []
    depths = []
    for k in range(1, n_bins + 1):
        cut = int(np.ceil(n * k / n_bins))
        segment_rate = y_sorted[:cut].mean()
        lifts.append(segment_rate / max(baseline, 1e-9))
        depths.append(k / n_bins)

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(depths, lifts, marker="o", label="Cumulative lift vs depth")
    ax.axhline(1.0, color="gray", linestyle="--", linewidth=1, label="Baseline")
    ax.set_xlabel("Fraction of population targeted (top scores first)")
    ax.set_ylabel("Lift vs random")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


@dataclass
class TargetingSimulationResult:
    baseline_conversion: float
    targeted_conversion: float
    random_mean_conversion: float
    random_std_conversion: float
    incremental_lift_vs_random: float
    incremental_lift_vs_random_pct: float
    budget_fraction: float


def simulate_acquisition_targeting(
    acquisition_prob: np.ndarray,
    ranking_scores: np.ndarray,
    top_fraction: float,
    n_random_trials: int,
    rng: np.random.Generator,
) -> TargetingSimulationResult:
    n = len(acquisition_prob)
    k = max(1, int(n * top_fraction))
    baseline = float(acquisition_prob.mean())

    top_idx = np.argsort(-ranking_scores)[:k]
    targeted_rate = float(acquisition_prob[top_idx].mean())

    random_rates = []
    for _ in range(n_random_trials):
        rand_idx = rng.choice(n, size=k, replace=False)
        random_rates.append(float(acquisition_prob[rand_idx].mean()))
    random_rates_arr = np.asarray(random_rates, dtype=float)
    random_mean = float(random_rates_arr.mean())
    random_std = float(random_rates_arr.std(ddof=1)) if len(random_rates_arr) > 1 else 0.0

    lift_vs_random = targeted_rate - random_mean
    lift_pct = (targeted_rate / max(random_mean, 1e-9) - 1.0) * 100.0

    return TargetingSimulationResult(
        baseline_conversion=baseline,
        targeted_conversion=targeted_rate,
        random_mean_conversion=random_mean,
        random_std_conversion=random_std,
        incremental_lift_vs_random=lift_vs_random,
        incremental_lift_vs_random_pct=lift_pct,
        budget_fraction=top_fraction,
    )


def format_metrics_table(metrics: Dict[str, float]) -> str:
    lines = []
    for k, v in metrics.items():
        lines.append(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")
    return "\n".join(lines)


In [ ]:
metrics = churn_discrimination_metrics(split.y_test, y_score_full_test)

plot_churn_model_performance(
    split.y_test,
    y_score_full_test,
    OUT_DIR / 'churn_model_performance.png',
    title='Churn model (held-out test set)',
)

pool_size = int(CFG['targeting'].get('acquisition_pool_size', len(split.X_test)))
pool_size = min(pool_size, len(split.X_test))
pool_idx = rng.choice(len(split.X_test), size=pool_size, replace=False)
X_pool = split.X_test.iloc[pool_idx].reset_index(drop=True)
acq_pool = split.acquisition_prob_test[pool_idx]

y_score_pool = predict_churn_proba(churn_model, X_pool)

clv_cfg = CLVModelConfig(
    margin_rate=float(CFG['clv']['margin_rate']),
    discount_monthly=float(CFG['clv']['discount_monthly']),
    max_horizon_months=int(CFG['clv']['max_horizon_months']),
    min_churn_prob=float(CFG['clv']['min_churn_prob']),
    max_churn_prob=float(CFG['clv']['max_churn_prob']),
)

pred_clv = estimate_clv(X_pool, y_score_pool, clv_cfg).to_numpy()
plot_clv_distribution(pred_clv, OUT_DIR / 'clv_distribution.png', title='Predicted CLV (test pool)')

top_fraction = float(CFG['targeting']['top_fraction'])
n_random_trials = int(CFG['targeting']['n_random_trials'])

sim = simulate_acquisition_targeting(
    acquisition_prob=acq_pool,
    ranking_scores=pred_clv,
    top_fraction=top_fraction,
    n_random_trials=n_random_trials,
    rng=rng,
)

converted = (rng.uniform(size=len(acq_pool)) < acq_pool).astype(int)
plot_lift_chart(
    converted,
    pred_clv,
    OUT_DIR / 'lift_chart.png',
    title='Acquisition lift by CLV targeting (test cohort)',
)

payload = {
    'churn_metrics': {'roc_auc': metrics.roc_auc, 'average_precision': metrics.average_precision},
    'targeting': {
        'baseline_conversion': sim.baseline_conversion,
        'targeted_conversion': sim.targeted_conversion,
        'random_mean_conversion': sim.random_mean_conversion,
        'random_std_conversion': sim.random_std_conversion,
        'incremental_lift_vs_random_pct': sim.incremental_lift_vs_random_pct,
        'top_fraction': top_fraction,
        'acquisition_pool_size': pool_size,
    },
    'clv_summary': {
        'mean': float(pred_clv.mean()),
        'p50': float(np.percentile(pred_clv, 50)),
        'p90': float(np.percentile(pred_clv, 90)),
    },
}
(OUT_DIR / 'metrics.json').write_text(json.dumps(payload, indent=2), encoding='utf-8')

print('=== Churn model (test) ===')
print(format_metrics_table({'roc_auc': metrics.roc_auc, 'pr_auc': metrics.average_precision}))

print('\n=== CLV (pool) ===')
print(format_metrics_table(payload['clv_summary']))

print('\n=== Acquisition targeting simulation (test pool) ===')
print(f"Budget (fraction targeted): {top_fraction:.0%}")
print(f"Baseline mean latent conversion: {sim.baseline_conversion:.4f}")
print(f"Targeted (top CLV) mean latent conversion: {sim.targeted_conversion:.4f}")
print(f"Random same-size cohort (mean ± std over trials): {sim.random_mean_conversion:.4f} ± {sim.random_std_conversion:.4f}")
print(
    f"Incremental lift vs random: {sim.incremental_lift_vs_random:.4f} ({sim.incremental_lift_vs_random_pct:+.2f}%)"
)

print('\nSaved figures + metrics.json to:', OUT_DIR)